## Vehicle sound classification (machine learning)
Dataset: https://www.kaggle.com/datasets/brinkor/vehicle-type-sound-dataset (public, no API key needed).
<br> Only the files that are used are downloaded (~250 MB of the 885 MB) into `data/project_3/case_1`.
<br> Outputs: `result/project_3/case_1/` (plots, metrics) and `.../model/` (best model).

In [ ]:
import sys

sys.path.insert(0, ".")
import base
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Audio
from sklearn.model_selection import train_test_split

In [ ]:
c3, SR = base.Case(3, 1), 22050
N_MAX = 250  # clips kept for each of the two large classes; the small class (2) is always kept completely
PREFIX = "vehicle_type_sound_dataset/"
HANDLE = "brinkor/vehicle-type-sound-dataset"
LABELS = [
    "class_0",
    "class_1",
    "motorcycle",
]  # class 2 = motorcycle

(labels_csv,) = base.fetch_kaggle_files(HANDLE, [PREFIX + "labels.csv"], c3.data)
df = pd.read_csv(labels_csv)
print("full dataset:", df["class"].value_counts().to_dict())

df = pd.concat(
    [
        g.sample(min(len(g), N_MAX), random_state=base.SEED) if c < 2 else g
        for c, g in df.groupby("class")
    ]
).reset_index(drop=True)

df["path"] = base.fetch_kaggle_files(
    HANDLE, [PREFIX + f for f in df.file_path], c3.data
)

print("used:", df["class"].value_counts().to_dict())

In [ ]:
# Split first (stratified), augment afterwards and only the training motorcycle clips
train, test = train_test_split(
    df, test_size=0.25, stratify=df["class"], random_state=base.SEED
)
train, test = train.reset_index(drop=True), test.reset_index(drop=True)

fig, ax = plt.subplots(2, 3, figsize=(13, 5))
for c in range(3):
    y = base.load_audio(train[train["class"] == c].path.iloc[0], SR)
    librosa.display.waveshow(y, sr=SR, ax=ax[0, c])
    ax[0, c].set_title(f"{LABELS[c]}: waveform")
    librosa.display.specshow(
        librosa.power_to_db(librosa.feature.melspectrogram(y=y, sr=SR), ref=np.max),
        sr=SR,
        x_axis="time",
        y_axis="mel",
        ax=ax[1, c],
    )
    
c3.save_fig("eda")

In [ ]:
# Features (cached). The minority class is augmented 5x (pitch x2, time stretch, noise, gain) in the training set only.
cache_tr, cache_te = (
    c3.data / f"feat_train_{N_MAX}.csv",
    c3.data / f"feat_test_{N_MAX}.csv",
)
if cache_tr.exists() and cache_te.exists():
    f_tr, f_te = pd.read_csv(cache_tr), pd.read_csv(cache_te)
else:
    f_tr = base.build_features(
        train.path,
        SR,
        augment=base.augment_clip,
        augment_mask=(train["class"] == 2).values,
    )
    f_te = base.build_features(test.path, SR)
    f_tr.to_csv(cache_tr, index=False), f_te.to_csv(cache_te, index=False)

groups, y_train, y_test = (
    f_tr.source.values,
    train["class"].values[f_tr.source],
    test["class"].values,
)
X_train, X_test = f_tr.drop(columns="source"), f_te.drop(columns="source")

pd.DataFrame(
    {
        "original train": train["class"].value_counts(),
        "train + augmented": pd.Series(y_train).value_counts(),
        "test": test["class"].value_counts(),
    }
).rename(index=dict(enumerate(LABELS))).plot.bar(rot=0, title="Samples per class")

c3.save_fig("class_balance")

Five models are tuned with 5-fold stratified *grouped* CV (macro-F1) on the training set, then scored once on the untouched test set.
The best model (highest CV macro-F1, faster predictor on ties) is saved in `model/`.

In [ ]:
res, best = base.compare_models(
    c3, X_train, y_train, X_test, y_test, LABELS, groups=groups
)

res.round(3)

Inference example with the saved model

In [ ]:
for p, true in test.groupby("class").head(1)[["path", "class"]].values:
    print(f"{p.name} (true: {LABELS[true]}) ->", base.predict_clip(c3, p, SR))
    
Audio(base.load_audio(p, SR), rate=SR)